In [1]:
import csv
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold

# -------------------------------------------------------------------------
# Step 1: Pre-calculate Train Statistics for Leak-Free Imputation
# -------------------------------------------------------------------------
train_raw = pd.read_csv("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/student_dataset.csv")

numeric_cols = [
    "ambient_temp",
    "process_temp",
    "coolant_flow",
    "shaft_speed",
    "drive_torque",
    "power_load",
    "component_wear",
    "thermal_stress",
    "mechanical_load",
    "vibration_index",
]

# Baseline medians and mode
imputation_medians = train_raw[numeric_cols].median().to_dict()
unit_mode = train_raw["unit_type"].mode()[0]
unit_type_mapping = {"S": 0, "M": 1, "H": 2}


# -------------------------------------------------------------------------
# Step 2: Row-by-Row Preprocessing & Domain Feature Engineering
# -------------------------------------------------------------------------
def process_dataset_row_by_row(
    file_path: str, is_train: bool = True
) -> pd.DataFrame:
    processed_rows = []

    with open(file_path, mode="r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            clean_record = {"unit_id": row.get("unit_id", "")}

            # 1. Categorical parsing & encoding with mode imputation
            raw_unit = row.get("unit_type", "").strip().upper()
            clean_unit = (
                raw_unit if raw_unit in unit_type_mapping else unit_mode
            )
            clean_record["unit_type_enc"] = unit_type_mapping[clean_unit]

            # 2. Continuous feature casting & median imputation
            for col in numeric_cols:
                val_str = row.get(col, "").strip()
                if val_str == "" or val_str.lower() == "nan":
                    clean_record[col] = float(imputation_medians[col])
                else:
                    try:
                        clean_record[col] = float(val_str)
                    except ValueError:
                        clean_record[col] = float(imputation_medians[col])

            # 3. Domain feature engineering
            clean_record["temp_diff"] = (
                clean_record["process_temp"] - clean_record["ambient_temp"]
            )
            clean_record["calculated_power"] = (
                clean_record["shaft_speed"] * clean_record["drive_torque"]
            ) / 9550.0
            clean_record["wear_vibe_ratio"] = (
                clean_record["component_wear"] * clean_record["vibration_index"]
            )
            clean_record["thermal_coolant_ratio"] = clean_record[
                "thermal_stress"
            ] / (clean_record["coolant_flow"] + 1e-5)

            # 4. Target label extraction
            if is_train and "condition" in row and row["condition"] != "":
                clean_record["condition"] = int(float(row["condition"]))

            processed_rows.append(clean_record)

    return pd.DataFrame(processed_rows)


print("Preprocessing training and testing datasets row-by-row...")
train_df = process_dataset_row_by_row("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/student_dataset.csv", is_train=True)
test_df = process_dataset_row_by_row("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/test.csv", is_train=False)

feature_columns = [
    c for c in train_df.columns if c not in ["unit_id", "condition"]
]
X = train_df[feature_columns].copy()
y = train_df["condition"].values
X_test = test_df[feature_columns].copy()

print(f"Features: {feature_columns}")
print(f"Train Shape: {X.shape} | Test Shape: {X_test.shape}")

# -------------------------------------------------------------------------
# Step 3: Stratified 5-Fold Soft-Voting Ensemble (HistGBM + RF)
# -------------------------------------------------------------------------
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_probabilities = np.zeros((len(X), 3))
test_probabilities = np.zeros((len(X_test), 3))

print("\nStarting Cross-Validation & Ensemble Training...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train_f, y_train_f = X.iloc[train_idx], y[train_idx]
    X_val_f, y_val_f = X.iloc[val_idx], y[val_idx]

    # Model 1: High-capacity Gradient Booster
    model_hgb_deep = HistGradientBoostingClassifier(
        max_iter=350,
        learning_rate=0.04,
        max_leaf_nodes=45,
        min_samples_leaf=20,
        random_state=42 + fold,
    )

    # Model 2: Conservative L2-Regularized Gradient Booster
    model_hgb_reg = HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.06,
        max_leaf_nodes=25,
        l2_regularization=1.0,
        random_state=100 + fold,
    )

    # Model 3: Variance-Reducing Random Forest
    model_rf = RandomForestClassifier(
        n_estimators=250,
        max_depth=16,
        min_samples_leaf=3,
        random_state=42 + fold,
        n_jobs=-1,
    )

    # Fit all three models
    model_hgb_deep.fit(X_train_f, y_train_f)
    model_hgb_reg.fit(X_train_f, y_train_f)
    model_rf.fit(X_train_f, y_train_f)

    # Soft-voting probability weighting
    val_probs = (
        0.45 * model_hgb_deep.predict_proba(X_val_f)
        + 0.35 * model_hgb_reg.predict_proba(X_val_f)
        + 0.20 * model_rf.predict_proba(X_val_f)
    )
    oof_probabilities[val_idx] = val_probs

    # Accumulate fold predictions for the test set
    test_probs = (
        0.45 * model_hgb_deep.predict_proba(X_test)
        + 0.35 * model_hgb_reg.predict_proba(X_test)
        + 0.20 * model_rf.predict_proba(X_test)
    )
    test_probabilities += test_probs / N_SPLITS

    fold_acc = accuracy_score(y_val_f, np.argmax(val_probs, axis=1))
    print(f"  Fold {fold} — Validation Accuracy: {fold_acc:.4f}")

# -------------------------------------------------------------------------
# Step 4: Model Evaluation
# -------------------------------------------------------------------------
oof_predictions = np.argmax(oof_probabilities, axis=1)
total_acc = accuracy_score(y, oof_predictions)
total_f1 = f1_score(y, oof_predictions, average="macro")

print("\n" + "=" * 50)
print(f"OVERALL OUT-OF-FOLD ACCURACY: {total_acc:.4f}")
print(f"OVERALL OUT-OF-FOLD MACRO F1: {total_f1:.4f}")
print("=" * 50)
print("\nClassification Report:\n", classification_report(y, oof_predictions))

# -------------------------------------------------------------------------
# Step 5: Generate Kaggle Submission File
# -------------------------------------------------------------------------
final_predictions = np.argmax(test_probabilities, axis=1)

submission = pd.DataFrame(
    {"unit_id": test_df["unit_id"], "condition": final_predictions}
)

submission.to_csv("/Users/kushagra/Downloads/industrial-machine-condition-classification-mtech/submission.csv", index=False)
print("File successfully created: 'submission.csv'")
print("\nSubmission preview:")
print(submission.head(10))

Preprocessing training and testing datasets row-by-row...
Features: ['unit_type_enc', 'ambient_temp', 'process_temp', 'coolant_flow', 'shaft_speed', 'drive_torque', 'power_load', 'component_wear', 'thermal_stress', 'mechanical_load', 'vibration_index', 'temp_diff', 'calculated_power', 'wear_vibe_ratio', 'thermal_coolant_ratio']
Train Shape: (16000, 15) | Test Shape: (4000, 15)

Starting Cross-Validation & Ensemble Training...
  Fold 1 — Validation Accuracy: 0.8825
  Fold 2 — Validation Accuracy: 0.8825
  Fold 3 — Validation Accuracy: 0.8878
  Fold 4 — Validation Accuracy: 0.8819
  Fold 5 — Validation Accuracy: 0.8831

OVERALL OUT-OF-FOLD ACCURACY: 0.8836
OVERALL OUT-OF-FOLD MACRO F1: 0.8731

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.94      0.91      8000
           1       0.88      0.83      0.86      4000
           2       0.88      0.82      0.85      4000

    accuracy                           0.88     16000
   